In [ ]:
import pandas as pd
import numpy as np
#1.计算各环节流失率
#1.1.1导入数据并转换value数据类型
item_properties_part1 = pd.read_csv('C:\\Users\\16225\\Desktop\\SQL\\Python\\真实kaggle案例\\item_properties_part1.csv')
item_properties_part2 = pd.read_csv('C:\\Users\\16225\\Desktop\\SQL\\Python\\真实kaggle案例\\item_properties_part2.csv')
item_properties = pd.concat([item_properties_part1,item_properties_part2],axis = 0)
events = pd.read_csv('C:\\Users\\16225\\Desktop\\SQL\\Python\\真实kaggle案例\\events.csv')
category_tree = pd.read_csv('C:\\Users\\16225\\Desktop\\SQL\\Python\\真实kaggle案例\\category_tree.csv')

def clean_value(value):
    if not isinstance(value,str):
        try:
            return float(value)
        except (ValueError,TypeError):
            return np.nan
    value.strip()
    if value.startswith('n'):
        clean_val = value[1:]
        try:
            return float(clean_val)
        except ValueError:
            return np.nan
    else:
        try:
            return float(value)
        except ValueError:
            return np.nan


item_category = item_properties.loc[(item_properties['property']=='categoryid'),['timestamp','itemid','property','value']]
item_category['value'] = item_category['value'].apply(clean_value)



#1.1.2将各产品所属类别根据时间戳merge进events文档中：
item_category = item_category[['timestamp', 'itemid', 'value']] \
                           .rename(columns={'value': 'categoryid'})
item_category['categoryid'] = item_category['categoryid'].fillna(-1).astype(str)
events_sorted = events.sort_values('timestamp')
item_category_sorted = item_category.sort_values('timestamp')

events_category = pd.merge_asof(
    events_sorted,
    item_category_sorted,
    on = 'timestamp',
    by = 'itemid',
    direction = 'backward',
    allow_exact_matches = True
)
events_category['datetime'] = pd.to_datetime(events_category['timestamp'],unit = 'ms')
#1.2清除categoryid无法成功匹配的行
events_category_clean = events_category.dropna(subset = 'categoryid').reset_index()
#1.2.1新增按年月格式的时间列
events_category_clean['year_m'] = events_category_clean['datetime'].dt.strftime('%Y-%m')

#1.3按时间、品类的留存率计算
events_time_category_desc = (events_category_clean.groupby(['year_m','categoryid']).agg(
    该类别下的item数量 = ('itemid','nunique'),
    浏览人数 = ('event',lambda x: (x == 'view').sum()),
    加购人数 = ('event',lambda x: (x == 'addtocart').sum()),
    交易人数 = ('event',lambda x: (x == 'transaction').sum())
).reset_index().assign(
    浏览_加购留存率 = lambda events_category_clean: np.where(events_category_clean['浏览人数']>0,events_category_clean['加购人数']/events_category_clean['浏览人数'],0),
    加购_交易留存率 = lambda events_category_clean: np.where(events_category_clean['加购人数']>0,events_category_clean['交易人数']/events_category_clean['加购人数'],0),
    浏览_交易留存率 = lambda events_category_clean: np.where(events_category_clean['浏览人数']>0,events_category_clean['交易人数']/events_category_clean['浏览人数'],0)
))


#1.4仅按时间留存率计算
events_time_desc = (events_category_clean.groupby('year_m').agg(
    浏览人数 = ('event',lambda x:(x=='view').sum()),
    加购人数 = ('event',lambda x:(x=='addtocart').sum()),
    交易人数 = ('event',lambda x:(x=='transaction').sum()),
    商品类别数量 = ('categoryid','nunique')
).reset_index().assign(
    浏览_加购留存率 = lambda events_category_clean:np.where(events_category_clean['浏览人数']>0,events_category_clean['加购人数']/events_category_clean['浏览人数'],0),
    加购_交易留存率 = lambda events_category_clean:np.where(events_category_clean['加购人数']>0,events_category_clean['交易人数']/events_category_clean['加购人数'],0)
))

#1.5计算按月分类的各item数据：
events_time_item_desc = (events_category_clean.groupby('itemid').agg(
    浏览次数 = ('event',lambda x:(x=='view').sum()),
    加购次数 = ('event',lambda x:(x=='addtocart').sum()),
    交易次数 = ('event',lambda x:(x=='transaction').sum())
).reset_index().assign(
    浏览_加购转化率 = lambda events_category_clean:np.where(events_category_clean['浏览次数']>0,events_category_clean['加购次数']/events_category_clean['浏览次数'],0),
    加购_交易转化率 = lambda events_category_clean:np.where(events_category_clean['加购次数']>0,events_category_clean['交易次数']/events_category_clean['加购次数'],0),
    浏览_交易转化率 = lambda events_category_clean:np.where(events_category_clean['浏览次数']>0,events_category_clean['交易次数']/events_category_clean['浏览次数'],0),
    交易次数是否大于浏览次数 = lambda events_category_clean:np.where(events_category_clean['交易次数']>events_category_clean['浏览次数'],1,0)
))
#1.6.1计算购买行为更多源自外源流量的商品比例：结果为0.008730218779282609%
tran_view_ratio = (events_time_item_desc['交易次数是否大于浏览次数'].sum())/(events_time_item_desc['itemid'].nunique())*100

#1.7 找出各产品所属大类：
category_tree = pd.read_csv('C:\\Users\\16225\\Desktop\\SQL\\Python\\真实kaggle案例\\category_tree.csv',names=['categoryid', 'parentid'])
parent = category_tree[category_tree['parentid'].isna()]['categoryid'].tolist()
#1.7.1自定义函数，用于找出category_tree.csv中各categoryid所对应的根大类（parentid）：
def find_the_parent(cid,df,top_list):
    if cid in top_list:
        return cid
    top = df[df['categoryid']==cid]['parentid'].values
    if len(top) == 0 or pd.isna(top[0]):
        return None
    return find_the_parent(top[0],df,top_list)
#1.7.2创建字典（categoryid:parentid），并统一parent_id的数据类型，进行merge
match_parent = {cid:find_the_parent(cid,category_tree,parent) for cid in category_tree['categoryid'].unique()}
match_parent_series = pd.Series(match_parent,name = 'parent_id')
match_parent_series = match_parent_series.reset_index()
match_parent_series.columns = ['categoryid','parent_id']
events_category_clean['categoryid'] = pd.to_numeric(events_category_clean['categoryid'], errors='coerce')
events_category_clean['categoryid'] = events_category_clean['categoryid'].astype('Int64')
match_parent_series['categoryid'] = pd.to_numeric(match_parent_series['categoryid'], errors='coerce')
match_parent_series['categoryid'] = match_parent_series['categoryid'].astype('Int64')
events_category_clean = events_category_clean.merge(match_parent_series,on='categoryid',how='left')

#1.7.3计算各大类按月份分类的转化率
events_time_parent = (events_category_clean.groupby(['year_m','parent_id_y']).agg(
    浏览次数 = ('event',lambda x:(x=='view').sum()),
    加购次数 = ('event',lambda x:(x=='addtocart').sum()),
    交易次数 = ('event',lambda x:(x=='transaction').sum()),
).reset_index().assign(
    浏览_加购转化率 = lambda x:np.where(x['浏览次数']>0,x['加购次数']/x['浏览次数'],0),
    加购_交易转化率 = lambda x:np.where(x['加购次数']>0,x['交易次数']/x['加购次数'],0),
    浏览_交易转化率 = lambda x:np.where(x['浏览次数']>0,x['交易次数']/x['浏览次数'],0)
))

#2.1计算各用户RF及转化率：
user_info =( events_category_clean.groupby('visitorid').agg(
    Recent = ('datetime','max'),
    Frequency = ('event','count'),
    浏览次数 = ('event',lambda x:(x == 'view').sum()),
    加购次数 = ('event',lambda x:(x == 'addtocart').sum()),
    交易次数 = ('event',lambda x:(x == 'transaction').sum())
).reset_index().assign(
    浏览_加购转化率 = lambda events_category_clean:np.where(events_category_clean['浏览次数']>0,events_category_clean['加购次数']/events_category_clean['浏览次数'],0),
    加购_交易转化率 = lambda events_category_clean:np.where(events_category_clean['加购次数']>0,events_category_clean['交易次数']/events_category_clean['加购次数'],0),
    浏览_交易转化率 = lambda events_category_clean:np.where(events_category_clean['浏览次数']>0,events_category_clean['交易次数']/events_category_clean['浏览次数'],0)
))
user_info.sort_values('浏览_交易转化率',ascending = False)

#3导出数据为csv文件
item_properties.to_csv('item_properties.csv',index = False)
events_category_clean.to_csv('events_category_clean.csv',index = False)
category_tree.to_csv('category_tree.csv',index = False)
events_time_category_desc.to_csv('events_time_category_desc.csv',index = False)
events_time_item_desc.to_csv('events_time_item_desc.csv',index = False)
events_time_parent.to_csv('events_time_parent.csv',index = False)
user_info.to_csv('user_info.csv',index = False)